# Train a model to compromise a multi-service company with TRL GRPO + LoRA

OpenRange admits **content-addressed, evolving** agent-training worlds. This notebook
trains a small model on the *cyber* pack's flagship: a **believable company** — a public
storefront in a dmz, plus internal APIs and databases on a segmented network. The flag is
a production-style credential **confined to an internal service**, with no front door. To
get it, the agent must **recon the estate, find a foothold (an SSRF on the storefront),
and pivot dmz→internal** over the network — graded by the world itself, no static dataset
of "correct" attacks.

It runs on **real Docker**: each rollout boots **one container per service** on a real
segmented network (and falls back to an in-process emulation when Docker isn't there).
Same torch-free TRL seam, same `http_get` / `submit` tools you bring — only the world
got real, and it **hardens as the agent improves**.

## 1. Install

```bash
uv pip install "openrange-trl[train]"   # torch, transformers, trl, datasets, accelerate, peft
```

`import openrange` never needs this extra — only the live training path below does. The
**CONTAINER** path needs a running Docker engine; the in-process fallback needs nothing.

## 2. Admit the company world

`company: True` flips the procedural builder onto a segmented 6–8-service estate —
deterministic and LLM-free, so this snapshot is byte-identical on every machine. We also
pick the backing: real Docker if it's running, else the in-process emulation.

In [1]:
import shutil
import subprocess

from cyber_webapp import WebappPack
from openrange_pack_sdk import Backing, Snapshot

from openrange.core.admit import admit

pack = WebappPack()
snapshot = admit(
    pack,
    manifest={
        "world": {"goal": "recover the hidden flag from the internal estate"},
        "pack": {"id": "webapp"},
        "runtime": {"tick": {"mode": "off"}},
        "npc": [],
        "seed": 3,
        "company": True,
    },
    max_repairs=3,
)
assert isinstance(snapshot, Snapshot), snapshot


def docker_ok():
    if shutil.which("docker") is None:
        return False
    try:
        probe = subprocess.run(
            ["docker", "info"], capture_output=True, timeout=10, check=False
        )
    except Exception:
        return False
    return probe.returncode == 0


BACKING = Backing.CONTAINER if docker_ok() else Backing.PROCESS
print(f"world {snapshot.snapshot_id[-12:]}   backing={BACKING.name}")

world ce3e3fcec648   backing=CONTAINER


## 3. The estate

A read-only walk of the world graph — no runtime. A public **storefront** in the `dmz`;
the rest internal on an **isolated** network. The flag is a hidden credential confined to
an internal database service — the storefront is the only way in, so the pivot is
mandatory.

In [2]:
g = snapshot.graph
services = list(g.by_kind("service"))
public = next(s for s in services if s.attrs.get("exposure") == "public")
ssrf = next(v for v in g.by_kind("vulnerability") if v.attrs.get("kind") == "ssrf")
flag_host = ssrf.attrs["params"]["internal_host"]


def network_of(svc):
    edge = next(e for e in g.out_edges(svc.id, "connected_to"))
    return g.nodes[edge.dst].attrs["name"]


nets = sorted({network_of(s) for s in services})
print(f"{len(services)} services across networks {nets}\n")
for s in sorted(services, key=lambda s: s.attrs.get("exposure")):
    tag = ""
    if s.attrs.get("exposure") == "public":
        tag = "  <- public entry (the only way in)"
    elif s.attrs["name"] == flag_host:
        tag = "  <- holds the flag"
    print(
        f"  {s.attrs['name']:<16} {s.attrs.get('exposure'):<9} {network_of(s):<9}{tag}"
    )
print(f"\nflag: a hidden credential on '{flag_host}' (internal) — reachable only by")
print(f"pivoting from '{public.attrs['name']}' via the SSRF on its endpoint.")

8 services across networks ['dmz', 'internal']

  orders-db        internal  internal   <- holds the flag
  orders-api       internal  internal 
  users-db         internal  internal 
  billing-db       internal  internal 
  catalog-api      internal  internal 
  payments-api     internal  internal 
  ledger-db        internal  internal 
  storefront       public    dmz        <- public entry (the only way in)

flag: a hidden credential on 'orders-db' (internal) — reachable only by
pivoting from 'storefront' via the SSRF on its endpoint.


## 4. See the reward before you train

The reward is the world's held-out verdict over the **HTTP path** — three subgoals: reach
the entry, extract *something*, submit the *right* flag. Driving the env's tools by hand
maps three behaviors onto it (no model, no GPU). The richest is the **full breach**: reach
the storefront, **SSRF-pivot into the internal service, and exfiltrate the credential** —
every coordinate read from the world graph (no hardcoded secrets) via the pack's reference
solver.

In [3]:
import json
import re

from cyber_webapp.reference_solver import solve_chain
from openrange_trl import EpisodeEnv

from examples.tools import WEB_TOOLS
from openrange.core.episode import EpisodeService


def pentest_task(snap):
    return next(t for t in snap.tasks if t.meta.get("family") == "webapp.pentest")


def entry_url(snap):
    return snap.graph.nodes[pentest_task(snap).entrypoints[0]].attrs["public_url"]


def flag_from(body):
    try:
        obj = json.loads(body)
    except json.JSONDecodeError:
        obj = None
    if isinstance(obj, dict):
        for key in ("credential", "data", "flag", "secret"):
            if isinstance(obj.get(key), str):
                return obj[key]
    found = re.search(
        r"(?:ghp_|gho_|sk_live_|AKIA)[A-Za-z0-9_]+|[0-9a-f-]{32,36}", body
    )
    return found.group(0) if found else ""


def breach(env, snap):
    env.http_get(entry_url(snap))
    trace = solve_chain(snap.graph, lambda p: env.http_get(p).split(chr(10), 1)[-1])
    env.submit(json.dumps({"flag": flag_from(trace.terminal)}))


def grade(name, act):
    svc = EpisodeService(pack, f"or-runs/cyber/{name}", backing=BACKING)
    env = EpisodeEnv(
        service=svc, snapshots={snapshot.snapshot_id: snapshot}, tools=WEB_TOOLS
    )
    env.reset(snapshot_id=snapshot.snapshot_id, task_id=pentest_task(snapshot).id)
    act(env)
    env._finalize()
    print(f"{name:<26} reward={env.reward:.3f}  resolved={env.report.passed}")
    svc.close()


entry = entry_url(snapshot)
grade("reach the storefront", lambda e: e.http_get(entry))
grade(
    "reach + wrong flag",
    lambda e: (e.http_get(entry), e.submit(json.dumps({"flag": "guess"}))),
)
grade("recon + pivot + breach", lambda e: breach(e, snapshot))

reach the storefront       reward=0.333  resolved=False


reach + wrong flag         reward=0.667  resolved=False


recon + pivot + breach     reward=1.000  resolved=True


Three behaviors, three grades:

| behavior | reward | why |
|------|--------|-----|
| reach the storefront | **0.333** | `reached_endpoint` only |
| reach + wrong flag | **0.667** | `+ extracted_anything`, but wrong |
| recon + pivot + breach | **1.0** | breached: pivoted dmz→internal, exfiltrated the credential |

That `0.333 → 1.0` spread is what GRPO turns into a gradient. `matched_flag == success` — the
agent has to actually **submit the right credential**, which it can only get by pivoting,
not by poking the storefront. (This surface is asserted in `tests/test_cyber_company.py`.)

## 5. Wrap the company world as a TRL environment

The torch-free adapter exposes the three seams TRL needs — `build_grpo_dataset` (the
pentest prompt rows), `make_environment_factory` (one `EpisodeEnv` per rollout), and
`make_reward_func` (the held-out grader). We build them once here to show the wiring; §8
rebuilds the rows and factory per evolved world and reuses this `reward_func`.

With `backing=CONTAINER`, each rollout boots the world's **per-service containers on a
docker network**, with the storefront published on a host port the policy's `http_get`
reaches. `sandbox=False` — the in-process policy talks to the world over the wire with the
tools you bring (`http_get`, `submit`).

In [4]:
from datasets import Dataset
from openrange_trl import build_grpo_dataset, make_environment_factory, make_reward_func

num_generations = 2
rows = [
    r
    for r in build_grpo_dataset(snapshot, repeat=num_generations)
    if "pentest" in r["task_id"]
]
dataset = Dataset.from_list(rows)
environment_factory = make_environment_factory(
    pack,
    [snapshot],
    "or-runs/cyber/envs",
    tools=WEB_TOOLS,
    backing=BACKING,
    sandbox=False,
)
reward_func = make_reward_func()

## 6. Load the policy + a LoRA adapter

LoRA on a small instruct model — the base stays frozen (fits a laptop), GRPO updates only
the low-rank adapters. Bump `model_name` to a larger instruct model for rollouts strong
enough to actually land the pivot.

In [5]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10703.80it/s]

## 7. Configure GRPO

`num_generations` completions of the same prompt, each against its own freshly booted
company. `beta=0` drops the reference model; `use_vllm=False` generates through
transformers; `max_tool_calling_iterations` is raised to **8** because recon → pivot →
submit is several turns, not the one-shot of a single-service bug. `max_steps=1` makes one
call here **one training round** — §8 wraps rounds into the curriculum.

In [6]:
from trl import GRPOConfig

config = GRPOConfig(
    output_dir="or-runs/cyber/trainer",
    per_device_train_batch_size=num_generations,
    num_generations=num_generations,
    steps_per_generation=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=1e-4,
    beta=0.0,
    temperature=1.0,
    max_completion_length=256,
    max_tool_calling_iterations=8,
    use_vllm=False,
    log_completions=False,
    logging_steps=1,
    report_to="none",
    disable_tqdm=True,
    save_strategy="no",
    bf16=False,
    fp16=False,
)

## 8. Train — and the company hardens back

Training is a **loop**: each round runs GRPO on the current world, then `auto_evolve` reads
the group's reward spread and, when the world is being solved, **plants a new vulnerability**
on the estate — a decoy the agent must tell apart from the real path — re-admits it (so the
harder world is still provably solvable), and the next round trains on that. We gate
evolution to the pentest family the agent trains on, so every step changes the breach world.
One round is one `GRPOTrainer.train()`; here's a real one against the live containers:

In [7]:
from openrange_trl import reward_variance_policy
from trl import GRPOTrainer

from openrange.core.curriculum import auto_evolve
from openrange.training import episode_reward


def grpo_round(snap):
    rows = [
        r
        for r in build_grpo_dataset(snap, repeat=num_generations)
        if "pentest" in r["task_id"]
    ]
    factory = make_environment_factory(
        pack,
        [snap],
        f"or-runs/cyber/{snap.snapshot_id[-12:]}",
        tools=WEB_TOOLS,
        backing=BACKING,
        sandbox=False,
    )
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=[reward_func],
        args=config,
        train_dataset=Dataset.from_list(rows),
        processing_class=tokenizer,
        environment_factory=factory,
        peft_config=lora,
    )
    trainer.train()
    return [e.report for e in (trainer.environments or ()) if e.report is not None]


reports = grpo_round(snapshot)
print(
    "one real GRPO round → rewards",
    [round(episode_reward(r).scalar, 3) for r in reports],
)

{'loss': '0', 'grad_norm': '0', 'learning_rate': '0.0001', 'num_tokens': '1071', 'completions/mean_length': '164.5', 'completions/min_length': '73', 'completions/max_length': '256', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '73', 'completions/min_terminated_length': '73', 'completions/max_terminated_length': '73', 'tools/call_frequency': '0.5', 'tools/failure_frequency': '0', 'rewards/reward_func/mean': '0.3333', 'rewards/reward_func/std': '0', 'reward': '0.3333', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '4.401', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '22.88', 'epoch': '0.5'}
{'train_runtime': '23.07', 'train_samples_per_second': '0.087', 'train_steps_per_second': '0.043', 'train_loss': '0', 'epoch': '0.5'}
one real GRPO round → rewards [0.333, 0.333]


That round changed nothing: both rollouts scored the same `0.333`, so the spread is zero
and GRPO has no gradient to learn from. On a live world that `0.333` is essentially free —
the running storefront answers a readiness probe before the agent acts — and the 0.5B never
climbs above it: it doesn't pivot to the internal credential. A real climb needs a bigger
model on a GPU, where rollouts genuinely differ. To watch the **loop** itself — solve,
harden, repeat — we stand in the §4 reference breach for that competent policy and run
several rounds; each solve hardens the company. Swap `grpo_round` back in on a GPU and the
policy's own climb drives it.

In [8]:
def scripted_round(snap):
    factory = make_environment_factory(
        pack,
        [snap],
        f"or-runs/cyber/evolve-{snap.snapshot_id[-12:]}",
        tools=WEB_TOOLS,
        backing=BACKING,
        sandbox=False,
    )
    reports = []
    for _ in range(num_generations):
        env = factory()
        env.reset(snapshot_id=snap.snapshot_id, task_id=pentest_task(snap).id)
        breach(env, snap)
        env._finalize()
        reports.append(env.report)
        env.service.close()
    return reports


def evolve_meta(snap):
    return snap.lineage.get("_evolve") or snap.lineage.get("manifest", {}).get(
        "_evolve", {}
    )


def run_curriculum(snap, *, rounds, run_round):
    chain = [snap]
    for r in range(1, rounds + 1):
        reports = run_round(snap)
        rewards = [round(episode_reward(x).scalar, 3) for x in reports]
        evolved = auto_evolve(
            snap,
            *reports,
            pack=pack,
            policy=reward_variance_policy,
            gate=lambda _snap, mut: mut.family == "webapp.pentest",
        )
        meta = {} if evolved is None else evolve_meta(evolved)
        move = (
            "converged" if evolved is None else meta.get("note", meta.get("kind", ""))
        )
        print(
            f"round {r}: world {snap.snapshot_id[-12:]}  rewards={rewards}  →  {move}"
        )
        if evolved is None:
            break
        snap = evolved
        chain.append(snap)
    return chain


chain = run_curriculum(snapshot, rounds=3, run_round=scripted_round)

round 1: world ce3e3fcec648  rewards=[1.0, 1.0]  →  add broken_authz on ep_billing-db_0


round 2: world 998555639550  rewards=[1.0, 1.0]  →  add command_injection on ep_billing-db_0


round 3: world 54eb8c709473  rewards=[1.0, 1.0]  →  add credential_gated_flag on ep_billing-db_0


## 9. The curriculum is a lineage

Each round's world is a fresh, **admission-checked** child snapshot — `auto_evolve` mutates
the company's vuln surface, re-admits (so the harder world is still provably solvable), and
tags the child with its parent. The estate's *shape* is fixed at admit time; what evolves is
the attack surface around it. Training produces a **chain of worlds**, fully attributable.

In [9]:
for snap in chain:
    kinds = sorted(
        str(v.attrs.get("kind")) for v in snap.graph.by_kind("vulnerability")
    )
    parent = (evolve_meta(snap).get("parent_snapshot_id") or "")[-12:] or "root"
    print(f"{snap.snapshot_id[-12:]}  parent={parent:>12}  vulns={kinds}")

ce3e3fcec648  parent=        root  vulns=['config_disclosure', 'metadata_credential_leak', 'ssrf', 'xxe', 'xxe']
998555639550  parent=ce3e3fcec648  vulns=['broken_authz', 'config_disclosure', 'metadata_credential_leak', 'ssrf', 'xxe', 'xxe']
54eb8c709473  parent=998555639550  vulns=['broken_authz', 'command_injection', 'config_disclosure', 'metadata_credential_leak', 'ssrf', 'xxe', 'xxe']
40b1afc4324c  parent=54eb8c709473  vulns=['broken_authz', 'command_injection', 'config_disclosure', 'credential_gated_flag', 'metadata_credential_leak', 'ssrf', 'xxe', 'xxe']


## Where this goes

You watched an agent breach a **real, multi-service company** — recon the estate, pivot
dmz→internal over the network, exfiltrate a confined credential — and the gym **harden** in
response. Two things take it past a laptop:

- **A real climb.** Swap the reference breach for `grpo_round` + a larger instruct model on a
  GPU. Now the *policy's* own improvement drives the hardening, no stand-in.
- **Throughput.** Each rollout boots a whole company; TRL's `AsyncGRPOTrainer` pipelines many
  against a vLLM server with the same `make_environment_factory`. Both trainers reset worlds
  serially, so a pooled / fast world realization is the gym-side lever that keeps a big batch
  from stalling on container boots (#294).

Honest scope: a laptop-scale model doesn't complete the pivot — the reward surface and the
breach are real and proven (§4), but mastery needs GPU-scale compute. The deterministic world + reward tests live in `packs/cyber_webapp` +
`tests/test_cyber_company.py`.